# Tutorial 10: Annotation & Entity Context

This notebook is a **one-stop reference** for enriching biological entities with metadata from public databases using `embpy`. It covers four annotator classes, a text-knowledge adapter, and a cross-entity statistics section that compares coverage and agreement across data sources.

### The four annotators

| Entity | Class | External DBs | Quick call |
|---|---|---|---|
| Gene | `GeneAnnotator` | MyGene, GTEx, STRING, Open Targets, GWAS Catalog | `ga.annotate("TP53")` |
| Protein | `ProteinAnnotator` | UniProt, InterPro | `pa.annotate("TP53")` |
| Small molecule | `MoleculeAnnotator` | RDKit (local), ChEMBL, ChEBI, KEGG, PubChem/UniChem | `ma.annotate("aspirin")` |
| Cell line | `CellLineAnnotator` | Cellosaurus, DepMap/CCLE, Cell Model Passports | `cla.annotate("A549")` |

### Cross-entity universal adapter

| Class | What it does |
|---|---|
| `TextResolver` | Fetch a textual description for any of the above from 6 knowledge sources (UniProt, NCBI Gene, Wikipedia, PubChem, DrugBank, Cellosaurus) and combine them into a single paragraph that can be passed to a text encoder via `BioEmbedder.embed_description(...)`. |

### What this notebook covers

1. Gene annotation (pathways, tissues, PPI, diseases, GWAS)
2. Protein annotation (function, sites, GO, isoforms)
3. Small-molecule annotation (physicochemistry, bioactivities, cross-refs)
4. Cell-line annotation (tissue, disease, lineage, MSI) + text-embedding similarity
5. Text knowledge as a universal adapter (any entity -> text -> embedding)
6. Cross-entity statistics: coverage, missingness, distributions, cross-database agreement

> Many cells hit live public APIs. If a cell fails for a single entity it is reported inline and the loop continues; rerun on a stable network for full coverage.


## Start here: standardized embedding outputs

The primary user-facing path for new embedding work is now `BioEmbedder.embed(...)`. It normalizes direct inputs and CSV/TSV/Parquet paths, canonicalizes identifiers by entity type, and routes results to either AnnData or a table.

Key contract:

- genes use Ensembl gene IDs, molecules use canonical SMILES, proteins use UniProt accessions
- aliases such as symbols, names, or original user inputs are kept as metadata/columns
- generated embeddings never go into AnnData `.X`; they go to `.obsm`, `.varm`, or `.uns`
- standalone AnnData uses sparse placeholder `.X` only
- table output defaults to Parquet plus a `.meta.json` sidecar
- `harmonize_dim=...` applies per-result PCA before export and records provenance

The older model-specific examples below are still useful for lower-level control. Treat this section as the standard output shape to prefer when building pipelines or sharing results.


In [ ]:
# Annotation enriches entities; standardized embedding outputs keep canonical
# IDs as primary keys and store human-readable labels as aliases/metadata.
from embpy import BioEmbedder

RUN_STANDARDIZED_EMBED_DEMO = False

if RUN_STANDARDIZED_EMBED_DEMO:
    embedder = BioEmbedder(device="auto")

    annotated_ready = embedder.embed(
        ["TP53", "MYC", "EGFR"],
        entity_type="gene",
        model="esm2_650M",
        output="table",
    )
    display(annotated_ready[["gene_symbol"]].head())

In [ ]:
import logging
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import anndata as ad

import embpy.pl as epl

logging.basicConfig(level=logging.WARNING)
warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook", font_scale=1.0)
%matplotlib inline

# Small, shared entity panels used throughout the notebook so every API call
# stays fast enough to run end-to-end.
GENES = ["TP53", "BRCA1", "EGFR", "MYC", "KRAS"]
MOLECULES = ["aspirin", "caffeine", "ibuprofen", "dexamethasone", "metformin"]
CELLLINES = ["A549", "HeLa", "MCF7", "HCT116", "K562"]

## 1. Gene annotation -- `GeneAnnotator`

`GeneAnnotator` aggregates gene-level metadata from MyGene.info, GTEx, STRING-DB, Open Targets, and the GWAS Catalog. Every source can be queried individually or all at once via `annotate(...)`.


In [ ]:
from embpy.resources import GeneAnnotator

ga = GeneAnnotator()

# Reactome + KEGG pathways.
pathways = ga.get_pathways("TP53")
print(f"Reactome pathways: {len(pathways['reactome'])}")
print(f"KEGG pathways:     {len(pathways['kegg'])}")
for p in pathways["reactome"][:3]:
    print(f"  {p['id']}: {p['name']}")

In [ ]:
# Tissue expression (GTEx bulk RNA-seq, median TPM per tissue).
tissues = ga.get_tissue_expression("TP53")
print(f"Tissues with expression data: {len(tissues)}")
for t in tissues[:5]:
    print(f"  {t['tissue_name']:30s}  TPM={t['median_tpm']:.1f}")

In [ ]:
# PPI partners from STRING-DB (sorted by combined score).
partners = ga.get_protein_interactions("TP53", n_partners=5)
for p in partners:
    print(f"  {p['partner']:10s}  score={p['combined_score']}")

In [ ]:
# Disease associations from Open Targets (evidence-weighted score).
diseases = ga.get_disease_associations("TP53", top_n=5)
for d in diseases:
    print(f"  {d['disease_name']:30s}  score={d['score']:.3f}")

In [ ]:
# One-shot multi-source annotation.
full = ga.annotate("BRCA1", sources=["pathways", "diseases"])
print(f"Pathways: {sum(len(v) for v in full['pathways'].values())}")
print(f"Diseases: {len(full['disease_associations'])}")

## 2. Protein annotation -- `ProteinAnnotator`

`ProteinAnnotator` resolves a gene symbol (or UniProt accession) to the canonical UniProt record, with optional per-isoform lookups, and returns structured metadata, GO terms, and functional sites.


In [ ]:
from embpy.resources import ProteinAnnotator

pa = ProteinAnnotator()

result = pa.annotate("TP53", sources=["metadata", "function", "location", "sites"])

print(f"Protein: {result['metadata']['protein_name']}")
print(f"Reviewed: {result['metadata']['reviewed']}")
print(f"Length: {result['metadata']['sequence_length']} aa")
print(f"\nFunction: {result['function']['function'][0][:100]}...")
print("\nSubcellular locations:")
for loc in result["subcellular_location"]:
    print(f"  {loc['location']}")
print("\nFunctional sites:")
print(f"  Active sites:  {len(result['functional_sites']['active_sites'])}")
print(f"  Binding sites: {len(result['functional_sites']['binding_sites'])}")
print(f"  Motifs:        {len(result['functional_sites']['motifs'])}")

In [ ]:
# GO terms split by ontology.
result_go = pa.annotate("TP53", sources="go")
go = result_go["go_terms"]
print(f"Molecular Function: {len(go['molecular_function'])} terms")
print(f"Biological Process: {len(go['biological_process'])} terms")
print(f"Cellular Component: {len(go['cellular_component'])} terms")
for t in go["molecular_function"][:3]:
    print(f"  {t['id']}: {t['term']}")

In [ ]:
# Per-isoform annotations.
result_iso = pa.annotate("TP53", sources="isoforms")
for iso in result_iso.get("isoform_annotations", []):
    print(f"  {iso['id']}: {iso['name']} ({iso['sequence_status']})")

## 3. Small-molecule annotation -- `MoleculeAnnotator`

Physicochemical properties are computed locally with RDKit (no network). Bioactivities, cross-references, and mechanism of action come from ChEMBL, ChEBI, KEGG, and PubChem.


In [ ]:
from embpy.resources import MoleculeAnnotator

ma = MoleculeAnnotator()

# RDKit-computed descriptors -- runs offline, ~milliseconds.
props = ma.get_physicochemical_properties("Cn1c(=O)c2c(ncn2C)n(C)c1=O")  # caffeine
for k, v in props.items():
    print(f"  {k:25s}: {v}")

In [ ]:
# One-call annotation across all sources.
result = ma.annotate("aspirin", sources="all")

print(f"SMILES: {result['canonical_smiles']}")
print("\nPhysicochemical:")
for k in ["molecular_weight", "logp", "qed", "lipinski_violations"]:
    print(f"  {k}: {result['physicochemical'].get(k)}")

print(f"\nTargets: {len(result.get('targets', []))}")
for t in result.get("targets", [])[:5]:
    print(f"  {t['target_name']}")

print(f"\nCross-references: {result.get('cross_references', {})}")

In [ ]:
# Annotate an AnnData.obs['drug'] column in one call (vectorised over cells).
from embpy.tl import annotate_molecules

adata_mol = ad.AnnData(
    obs=pd.DataFrame(
        {"drug": ["aspirin", "caffeine", "ibuprofen", "aspirin"]},
        index=[f"cell_{i}" for i in range(4)],
    ),
)
adata_mol = annotate_molecules(adata_mol, column="drug", sources="structural")
print(adata_mol.obs[["drug", "mol_molecular_weight", "mol_logp", "mol_qed"]].to_string())

## 4. Cell-line annotation -- `CellLineAnnotator`

Pulls metadata for 150k+ cell lines across Cellosaurus, DepMap/CCLE, and Cell Model Passports. After annotation we build an `AnnData`, embed the cell-line text descriptions, and explore similarity with `embpy.pl`.


In [ ]:
from embpy.resources.cellline import CellLineAnnotator

cla = CellLineAnnotator(rate_limit_delay=0.5)

cl_annotations: dict[str, dict] = {}
for cl in CELLLINES:
    try:
        cl_annotations[cl] = cla.annotate(cl)
    except Exception as e:
        print(f"  {cl:10s}  FAILED: {e}")

cl_df = pd.DataFrame(
    [
        {
            "name": cl,
            "tissue": ann.get("tissue", ""),
            "disease": ann.get("disease", ""),
            "lineage": ann.get("lineage", ""),
            "species": ann.get("species", ""),
            "model_type": ann.get("model_type", ""),
            "msi_status": ann.get("msi_status", ""),
            "is_problematic": ann.get("is_problematic", False),
        }
        for cl, ann in cl_annotations.items()
    ]
)
cl_df

In [ ]:
# Text descriptions (for downstream text embedding).
for cl in list(cl_annotations)[:3]:
    text = cla.get_text_description(cl)
    print(f"\n{cl}:")
    print(f"  {text[:200]}..." if len(text) > 200 else f"  {text}")

In [ ]:
# Embed cell-line text descriptions.
from embpy.embedder import BioEmbedder

embedder = BioEmbedder(device="auto")

cl_embs: dict[str, np.ndarray] = {}
for cl in cl_annotations:
    try:
        cl_embs[cl] = embedder.embed_description(
            cl,
            model="minilm_l6_v2",
            entity_type="cellline",
        )
    except Exception as e:
        print(f"  {cl:10s}  FAILED: {e}")

embedded_cls = [cl for cl in CELLLINES if cl in cl_embs]
adata_cl = ad.AnnData(
    obs=pd.DataFrame(
        {
            "cell_line": embedded_cls,
            "tissue": [cl_annotations[cl].get("tissue", "") for cl in embedded_cls],
            "disease": [cl_annotations[cl].get("disease", "") for cl in embedded_cls],
            "lineage": [cl_annotations[cl].get("lineage", "") for cl in embedded_cls],
        },
        index=pd.Index(embedded_cls),
    ),
)
adata_cl.obsm["X_cellline_text"] = np.stack([cl_embs[cl] for cl in embedded_cls]).astype(np.float32)
print(adata_cl)

In [ ]:
# Cosine-similarity heatmap of cell-line text embeddings.
epl.plot_similarity_heatmap(
    adata=adata_cl,
    obsm_key="X_cellline_text",
    metric="cosine",
    labels=embedded_cls,
    title="Cell-line text embedding similarity",
    annot=True,
    fmt=".2f",
)
plt.show()

In [ ]:
# Hierarchical dendrogram.
epl.dendrogram(
    adata_cl,
    obsm_key="X_cellline_text",
    metric="cosine",
    linkage_method="average",
    title="Cell-line hierarchical clustering",
)
plt.show()

In [ ]:
# Annotate an AnnData with a 'cell_line' column in one call.
adata_demo = ad.AnnData(
    obs=pd.DataFrame(
        {
            "cell_line": ["A549", "A549", "MCF7", "MCF7", "HeLa"],
            "perturbation": ["TP53", "BRCA1", "EGFR", "MYC", "KRAS"],
        },
        index=[f"cell_{i}" for i in range(5)],
    ),
)
adata_demo = cla.annotate_adata(adata_demo, column="cell_line")
adata_demo.obs

## 5. Text knowledge as a universal adapter -- `TextResolver`

`TextResolver` fetches free-text descriptions for any gene / protein / molecule from 6 sources (UniProt, NCBI Gene, Wikipedia, PubChem, DrugBank, Cellosaurus). Combined with `BioEmbedder.embed_description(...)` this turns *any* of the annotators above into an embedding pipeline without model-specific plumbing.


In [ ]:
from embpy.resources.text import TextResolver

tr = TextResolver()

# Per-source descriptions for one gene.
descs = tr.get_gene_description("TP53")
for source, text in descs.items():
    if not text:
        continue
    print(f"\n[{source.upper()}] ({len(text)} chars)")
    print(f"  {text[:200]}{'...' if len(text) > 200 else ''}")

In [ ]:
# Combined paragraph that concatenates all available sources -- the preferred
# input for `BioEmbedder.embed_description`.
combined = tr.get_combined_description("TP53", entity_type="gene")
print(f"Combined description ({len(combined)} chars):\n")
print(combined[:500])

In [ ]:
# Same resolver works for proteins and molecules.
prot_descs = tr.get_protein_description("TP53")
mol_descs = tr.get_molecule_description("aspirin")

for source, text in list(prot_descs.items())[:3]:
    print(f"[PROT/{source.upper()}] {text[:120]}..." if text else f"[PROT/{source.upper()}] (empty)")
for source, text in list(mol_descs.items())[:3]:
    print(f"[MOL/{source.upper()}]  {text[:120]}..." if text else f"[MOL/{source.upper()}]  (empty)")

In [ ]:
# Embed every entity as text. Works across entity types.
text_embs: dict[str, np.ndarray] = {}
for entity in GENES + MOLECULES:
    try:
        text_embs[entity] = embedder.embed_description(entity, model="minilm_l6_v2")
    except Exception as e:
        print(f"  {entity:15s}  FAILED: {e}")

entities = [e for e in GENES + MOLECULES if e in text_embs]
entity_types = ["gene"] * sum(1 for g in GENES if g in text_embs) + ["molecule"] * sum(
    1 for d in MOLECULES if d in text_embs
)

adata_text = ad.AnnData(
    obs=pd.DataFrame(
        {"entity": entities, "type": entity_types},
        index=pd.Index(entities),
    ),
)
adata_text.obsm["X_text"] = np.stack([text_embs[e] for e in entities]).astype(np.float32)

epl.plot_similarity_heatmap(
    adata=adata_text,
    obsm_key="X_text",
    metric="cosine",
    labels=entities,
    title="Text embedding cosine similarity (genes + molecules)",
    annot=True,
    fmt=".2f",
)
plt.show()

In [ ]:
# embed_text() takes any raw string -- useful for hypotheses or custom notes.
custom_texts = [
    "TP53 is a tumor suppressor that responds to DNA damage.",
    "BRCA1 repairs double-strand DNA breaks via homologous recombination.",
    "Aspirin inhibits cyclooxygenase enzymes COX-1 and COX-2.",
]
for text in custom_texts:
    emb = embedder.embed_text(text, model="minilm_l6_v2")
    print(f'  "{text[:60]}..."  dim={emb.shape[0]}')

## 6. Cross-entity statistics

Every annotator returns a richly structured dict; this section turns those dicts into numbers you can compare across entities and across data sources. We cover:

- **6a.** Per-entity annotation counts (# pathways, # GO terms, # tissues, # diseases, # PPI partners)
- **6b.** Missingness heatmap (entities x annotation sources)
- **6c.** Distribution plots (GO terms / tissues per entity)
- **6d.** Cross-database agreement (STRING PPI vs Open Targets, ChEMBL vs PubChem)

The stats are computed on the small shared panels defined at the top; in production you would swap in your full dataset.


### 6a. Per-entity annotation counts

In [ ]:
# For every gene in the panel, pull a broad annotation set and count what
# each source returned.
stats_rows = []
for gene in GENES:
    try:
        g = ga.annotate(gene, sources=["pathways", "tissues", "interactions", "diseases"])
    except Exception:
        g = {}
    try:
        p = pa.annotate(gene, sources=["metadata", "go", "sites"])
    except Exception:
        p = {}
    go = p.get("go_terms", {})
    sites = p.get("functional_sites", {})
    stats_rows.append(
        {
            "gene": gene,
            "seq_len": (p.get("metadata") or {}).get("sequence_length"),
            "reactome": len((g.get("pathways") or {}).get("reactome", [])),
            "kegg": len((g.get("pathways") or {}).get("kegg", [])),
            "tissues": len(g.get("tissue_expression", [])),
            "ppi": len(g.get("protein_interactions", [])),
            "diseases": len(g.get("disease_associations", [])),
            "go_mf": len(go.get("molecular_function", [])),
            "go_bp": len(go.get("biological_process", [])),
            "go_cc": len(go.get("cellular_component", [])),
            "active_sites": len(sites.get("active_sites", [])),
            "binding_sites": len(sites.get("binding_sites", [])),
        }
    )
gene_stats = pd.DataFrame(stats_rows).set_index("gene")
gene_stats

In [ ]:
# Molecule-side counts: # targets, # bioassays, # cross-refs, # ChEBI roles.
mol_rows = []
for name in MOLECULES:
    try:
        r = ma.annotate(name, sources="all")
    except Exception:
        r = {}
    phys = r.get("physicochemical", {}) or {}
    mol_rows.append(
        {
            "molecule": name,
            "mw": phys.get("molecular_weight"),
            "logp": phys.get("logp"),
            "qed": phys.get("qed"),
            "lipinski_viol": phys.get("lipinski_violations"),
            "targets": len(r.get("targets", []) or []),
            "chebi_roles": len(r.get("chebi_roles", []) or []),
            "kegg_paths": len(r.get("kegg_pathways", []) or []),
            "xrefs": len(r.get("cross_references", {}) or {}),
        }
    )
mol_stats = pd.DataFrame(mol_rows).set_index("molecule")
mol_stats

### 6b. Missingness heatmap across entities x sources

This is the most useful "did my query succeed?" diagnostic: rows = entities, columns = annotation sources, cell value = *1 if at least one record was returned, 0 otherwise*. Patterns highlight entities with poor coverage (novel targets, orphan diseases) and sources that are systematically empty in your panel.


In [ ]:
# Build a single missingness matrix across all four entity types.
miss_rows: list[dict] = []

for gene, row in gene_stats.iterrows():
    miss_rows.append(
        {
            "entity": gene,
            "kind": "gene",
            "pathways": int(row["reactome"] > 0 or row["kegg"] > 0),
            "tissues": int(row["tissues"] > 0),
            "ppi": int(row["ppi"] > 0),
            "diseases": int(row["diseases"] > 0),
            "go": int(row["go_mf"] + row["go_bp"] + row["go_cc"] > 0),
            "sites": int(row["active_sites"] + row["binding_sites"] > 0),
        }
    )

for mol, row in mol_stats.iterrows():
    miss_rows.append(
        {
            "entity": mol,
            "kind": "molecule",
            "pathways": int(row["kegg_paths"] > 0),
            "tissues": 0,
            "ppi": int(row["targets"] > 0),  # targets are PPI-like
            "diseases": 0,
            "go": 0,
            "sites": int(row["chebi_roles"] > 0),
        }
    )

for cl, ann in cl_annotations.items():
    miss_rows.append(
        {
            "entity": cl,
            "kind": "cellline",
            "pathways": 0,
            "tissues": int(bool(ann.get("tissue"))),
            "ppi": 0,
            "diseases": int(bool(ann.get("disease"))),
            "go": 0,
            "sites": int(bool(ann.get("lineage"))),
        }
    )

miss_df = pd.DataFrame(miss_rows).set_index("entity")
kinds = miss_df.pop("kind")

fig, ax = plt.subplots(figsize=(7, 0.4 * len(miss_df) + 1.5))
sns.heatmap(
    miss_df,
    cmap=sns.color_palette(["#f5f5f5", "#2c7bb6"], as_cmap=False),
    cbar=False,
    linewidths=0.5,
    linecolor="white",
    annot=False,
    ax=ax,
)
# Colour-code row labels by kind.
colors = {"gene": "#1b9e77", "molecule": "#d95f02", "cellline": "#7570b3"}
for tick, kind in zip(ax.get_yticklabels(), kinds, strict=False):
    tick.set_color(colors.get(kind, "black"))
ax.set_title("Annotation availability (blue = at least one record)")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

### 6c. Distribution plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))

gene_stats[["go_mf", "go_bp", "go_cc"]].plot(
    kind="bar",
    stacked=True,
    ax=axes[0],
    color=["#1f77b4", "#ff7f0e", "#2ca02c"],
)
axes[0].set_title("GO terms per gene (stacked)")
axes[0].set_ylabel("# terms")
axes[0].legend(loc="upper right", fontsize=8)

gene_stats["tissues"].plot(kind="bar", ax=axes[1], color="#2ca02c")
axes[1].set_title("GTEx tissues with expression")
axes[1].set_ylabel("# tissues")

mol_stats["targets"].plot(kind="bar", ax=axes[2], color="#d95f02")
axes[2].set_title("ChEMBL targets per molecule")
axes[2].set_ylabel("# targets")

for ax in axes:
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

### 6d. Cross-database agreement

Two quick consistency checks:

- **Genes:** are STRING PPI partners of TP53 enriched in its Open Targets disease-associated genes? (Same biology, different evidence types.)
- **Molecules:** how much overlap is there between ChEMBL-reported targets and the target field returned by PubChem for the same compound?

Both are "nice to have" smoke tests before you trust a single source in a downstream analysis.


In [ ]:
# Genes: STRING PPI vs Open Targets disease-associated genes (by inference --
# Open Targets returns diseases, so we fetch the top diseases and pull the
# *associated genes* for each; overlap with STRING partners gives a crude
# agreement signal on "genes in the TP53 neighbourhood".
from embpy.resources.gene.annotator import GeneAnnotator

string_partners = {p["partner"] for p in (ga.get_protein_interactions("TP53", n_partners=20) or [])}

# Open Targets only returns diseases directly, but we can look at the
# *other* genes that the same top diseases are associated with. We use the
# annotator's internal helper where available and fall back to the raw
# disease list when not.
ot_genes: set[str] = set()
try:
    diseases = ga.get_disease_associations("TP53", top_n=5) or []
    for d in diseases:
        # Some annotator versions return a list of associated genes per disease.
        for g in d.get("associated_genes") or []:
            ot_genes.add(g)
except Exception as e:
    print(f"Open Targets lookup failed: {e}")

overlap = string_partners & ot_genes
print(f"STRING partners:        {len(string_partners)}")
print(f"Open Targets co-genes:  {len(ot_genes)}")
print(f"Overlap:                {len(overlap)}  -> {sorted(overlap)[:10]}")
if string_partners or ot_genes:
    jaccard = len(overlap) / max(1, len(string_partners | ot_genes))
    print(f"Jaccard:                {jaccard:.3f}")

In [ ]:
# Molecules: ChEMBL targets vs PubChem/UniChem cross-ref targets.
chembl_targets: dict[str, set[str]] = {}
pubchem_xrefs: dict[str, set[str]] = {}

for name in MOLECULES:
    try:
        r = ma.annotate(name, sources="all")
    except Exception:
        r = {}
    chembl_targets[name] = {
        (t.get("target_name") or "").lower() for t in (r.get("targets") or []) if t.get("target_name")
    }
    xrefs = r.get("cross_references") or {}
    pubchem_xrefs[name] = set(xrefs.keys())

summary = pd.DataFrame(
    {
        "chembl_targets": [len(chembl_targets[m]) for m in MOLECULES],
        "pubchem_xrefs": [len(pubchem_xrefs[m]) for m in MOLECULES],
    },
    index=MOLECULES,
)
summary.plot(kind="bar", figsize=(8, 3))
plt.title("ChEMBL target count vs PubChem cross-reference count per molecule")
plt.ylabel("# records")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

summary

## Summary

| Capability | One-liner |
|---|---|
| Gene pathways | `GeneAnnotator().get_pathways(gene)` |
| Tissue expression | `GeneAnnotator().get_tissue_expression(gene)` |
| PPI partners | `GeneAnnotator().get_protein_interactions(gene)` |
| Disease associations | `GeneAnnotator().get_disease_associations(gene)` |
| Full gene annotation | `GeneAnnotator().annotate(gene, sources=...)` |
| Protein metadata / function / sites / GO / isoforms | `ProteinAnnotator().annotate(gene, sources=...)` |
| Molecule physicochemistry (local) | `MoleculeAnnotator().get_physicochemical_properties(smiles)` |
| Full molecule annotation | `MoleculeAnnotator().annotate(name, sources="all")` |
| Annotate `AnnData.obs` | `annotate_molecules(adata, column="drug")` |
| Cell-line metadata | `CellLineAnnotator().annotate(cell_line)` |
| Text description (any entity) | `TextResolver().get_combined_description(id, entity_type=...)` |
| Entity -> text embedding | `embedder.embed_description(id, model="minilm_l6_v2")` |
| Custom text embedding | `embedder.embed_text(text, model="minilm_l6_v2")` |

For weighted protein-embedding strategies (TPM-weighted isoform averaging, site-weighted residue pooling, expression-context concatenation), see [**Tutorial 3: Protein Embeddings**](03_protein_embeddings.ipynb#15-weighted-protein-embeddings) section 15.

**Next:** [Tutorial 11: Cross-Species Embeddings](11_cross_species_embeddings.ipynb)
